# 05 · Un harness in miniatura

Un **harness** è tutto ciò che circonda il modello per renderlo un agente affidabile.
Ne costruiamo una versione minima con quattro meccanismi che convivono:
1. **audit** delle chiamate ai tool (middleware);
2. **limite** al numero di tool call (sicurezza deterministica);
3. **approvazione umana** prima di un'azione sensibile;
4. **verifica** del risultato prima di concludere.

## Obiettivi, prerequisiti e modalità di lettura

Assemblerai tool con effetto, audit, limite e verifica. Durata: 25–35 minuti. L'email resta simulata: nessun messaggio viene inviato davvero.

Ogni blocco di codice è preceduto da una spiegazione e seguito da un **output
atteso**. Quando interviene un modello, l'output atteso descrive proprietà e
invarianti, non una frase letterale. Esegui le celle in ordine e non saltare i
casi negativi: mostrano il confine del meccanismo, non un incidente del corso.

## Setup (autonomo)

Ogni notebook è **indipendente**: non importa nulla dal progetto. Qui carichiamo la chiave
API dal file `.env` e creiamo un modello. Esegui le celle in ordine dall'alto verso il basso.

### Spiegazione del blocco · Setup

Il notebook valida ambiente e chiave prima di introdurre effetti collaterali simulati.

In [ ]:
# Carichiamo le variabili d'ambiente dal file `.env`.
# Lo cerchiamo nella cartella corrente e in quelle superiori, così il notebook
# funziona sia se avviato dalla radice del progetto sia dalla cartella `notebooks`.
import os
from pathlib import Path

from dotenv import load_dotenv


def trova_env() -> Path:
    for cartella in (Path.cwd(), *Path.cwd().resolve().parents):
        if (cartella / ".env").is_file():
            return cartella / ".env"
    raise FileNotFoundError("File .env non trovato: copia .env.example in .env e aggiungi la chiave.")


env_file = trova_env()
load_dotenv(env_file, override=False)          # carica le variabili senza sovrascrivere quelle già presenti
assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY mancante nel file .env"
print("Ambiente caricato da:", env_file)

### Output atteso

Percorso `.env` caricato.

### Spiegazione del blocco · Modello

Il client viene configurato una sola volta e non conserva lato provider la conversazione.

In [ ]:
# `ChatOpenAI` è il wrapper LangChain attorno al modello.
# Lo creiamo una volta e lo riusiamo in tutto il notebook.
from langchain_openai import ChatOpenAI

MODELLO = os.getenv("OPENAI_MODEL", "gpt-5.4-mini")   # modello economico, va bene per imparare
model = ChatOpenAI(
    model=MODELLO,
    use_responses_api=True,   # API "responses" di OpenAI
    store=False,              # non conservare la conversazione sui server OpenAI
)
print("Modello pronto:", MODELLO)

### Output atteso

Nome del modello pronto.

## 1 · Un tool con effetto collaterale

Simuliamo un'azione "sensibile": inviare una email. Non la spediamo davvero, la registriamo.

### Spiegazione del blocco · Effetto collaterale simulato

Il tool registra email in memoria invece di inviarle. Questo permette di studiare audit e verifica senza contattare servizi esterni.

In [ ]:
from langchain_core.tools import tool

INVIATE: list[dict] = []


@tool
def invia_email(destinatario: str, testo: str) -> str:
    """Invia una email (qui simulata) a un destinatario."""
    INVIATE.append({"a": destinatario, "testo": testo})
    return f"Email inviata a {destinatario}."

### Output atteso

Nessun output. `INVIATE` è vuota.

## 2 · Audit: un middleware attorno ai tool

Un middleware `AgentMiddleware` può avvolgere ogni chiamata a un tool. Lo usiamo per
**tracciare** cosa viene eseguito — senza registrare argomenti sensibili.

### Spiegazione del blocco · Audit middleware

Il middleware registra inizio e fine attorno alla vera esecuzione del tool. Non salva destinatario o testo, riducendo esposizione di dati sensibili.

In [ ]:
from langchain.agents.middleware import AgentMiddleware

TRACCIA: list[str] = []


class AuditMiddleware(AgentMiddleware):
    # `wrap_tool_call` viene chiamato per ogni tool: prima, dopo (o su errore).
    def wrap_tool_call(self, request, handler):
        nome = request.tool_call["name"]
        TRACCIA.append(f"start:{nome}")
        risultato = handler(request)      # esegue davvero il tool
        TRACCIA.append(f"done:{nome}")
        return risultato

### Output atteso

Nessun output. `TRACCIA` è vuota e verrà popolata durante la tool call.

## 3 · Limite di tool call

Un tetto deterministico evita loop infiniti o costi fuori controllo. LangChain offre un
middleware pronto: si ferma quando l'agente supera il numero di chiamate consentite.

### Spiegazione del blocco · Limite deterministico

Il limite non dipende dalla buona volontà del modello. Dopo cinque tool call il runtime termina il ciclo secondo la policy configurata.

In [ ]:
from langchain.agents.middleware import ToolCallLimitMiddleware

limite = ToolCallLimitMiddleware(run_limit=5, exit_behavior="end")   # max 5 tool call per run

### Output atteso

Nessun output. `limite` è pronto con `run_limit=5`.

## 4 · Approvazione umana (human-in-the-loop)

Per le azioni sensibili vogliamo che un umano approvi. Lo mostriamo in modo esplicito:
eseguiamo l'agente in *modalità simulazione* e vediamo cosa AVREBBE fatto, decidendo noi.
(LangChain supporta anche interruzioni automatiche con `interrupt_on`.)

### Spiegazione del blocco · Assemblaggio dell'harness minimo

Tool, audit e limite convivono nello stesso graph. Questo mostra che un harness è composizione di policy, non una singola classe.

In [ ]:
from langchain.agents import create_agent

agente = create_agent(
    model=model,
    tools=[invia_email],
    middleware=[AuditMiddleware(), limite],   # audit + limite attivi
    system_prompt="Quando l'utente chiede di scrivere a qualcuno, usa invia_email.",
)

### Output atteso

Nessun output. `agente` è invocabile.

### Spiegazione del blocco · Azione richiesta dall'utente

Il modello interpreta destinatario e contenuto, poi usa il tool. L'invio resta simulato ma segue lo stesso flusso di un effetto reale.

In [ ]:
esito = agente.invoke({"messages": [{
    "role": "user",
    "content": "Scrivi a mario@example.com un breve promemoria per la riunione di domani.",
}]})
print(esito["messages"][-1].text)

### Output atteso

Conferma dell'email a `mario@example.com`. La formulazione può variare.

## 5 · Verifica del risultato

Prima di fidarci, controlliamo con del **codice deterministico** che l'effetto ci sia stato.
La verifica non la fa il modello: la fa il nostro programma.

### Spiegazione del blocco · Verifica deterministica

Si controllano sia trace sia stato concreto. L'assert impedisce di considerare completato un run che non ha prodotto esattamente un effetto.

In [ ]:
# Traccia dei tool e prova concreta dell'effetto collaterale.
print("Traccia audit:", TRACCIA)
print("Email registrate:", INVIATE)
assert len(INVIATE) == 1, "Attesa esattamente una email"
print("Verifica superata: l'azione è avvenuta una sola volta.")

### Output atteso

Trace `start:invia_email`, `done:invia_email`, una email registrata e messaggio `Verifica superata`.

## Prova tu

- Abbassa `run_limit` a 1 e chiedi due email: vedrai il limite fermare l'agente.
- Nell'audit, aggiungi la durata di ogni tool (con `time.monotonic()`).

**Idea chiave**: l'harness è un insieme di *guardrail* attorno al modello — osservabilità,
limiti, approvazione e verifica — che trasformano un modello in un agente su cui fare affidamento.

## Laboratorio aggiuntivo

Gli esempi seguenti riusano quanto costruito sopra. Il primo amplia il caso normale; il
secondo esercita un confine, un errore o una proprietà che spesso causa bug reali.

## Esempio aggiuntivo: verifier riutilizzabile

### Spiegazione del blocco

Estrarre la verifica in una funzione consente di applicare lo stesso contratto a run diversi.

In [ ]:
def verifica_email(registro: list[dict], destinatario: str) -> tuple[bool, str]:
    corrispondenze = [item for item in registro if item["a"] == destinatario]
    if len(corrispondenze) != 1:
        return False, f"attese 1 email, trovate {len(corrispondenze)}"
    return True, "una sola email al destinatario corretto"

print(verifica_email(INVIATE, "mario@example.com"))
print(verifica_email(INVIATE, "assente@example.com"))

### Output atteso

Prima tupla `(True, ...)`; seconda `(False, 'attese 1 email, trovate 0')`.

## Esempio aggiuntivo: decisione HITL esplicita

### Spiegazione del blocco

La decisione umana deve essere un dato verificabile e legato all'azione, non una frase vaga nel prompt.

In [ ]:
def applica_decisione(azione: dict, decisione: str) -> str:
    if decisione not in {"approve", "reject"}:
        raise ValueError("decisione non valida")
    return "ESEGUITA" if decisione == "approve" else "BLOCCATA"

azione = {"tool": "invia_email", "destinatario": "mario@example.com"}
print("approve ->", applica_decisione(azione, "approve"))
print("reject  ->", applica_decisione(azione, "reject"))

### Output atteso

`approve -> ESEGUITA` e `reject -> BLOCCATA`.

## Riepilogo e troubleshooting

Prima di proseguire, prova a spiegare con parole tue: quale stato è cambiato, quale
componente ha preso la decisione e quale prova rende osservabile l'esito.

Se una cella fallisce:

1. rileggi l'output atteso e individua la prima invariante non rispettata;
2. verifica di aver eseguito tutte le celle precedenti nello stesso kernel;
3. per i notebook live, controlla `.env`, modello disponibile e quota API;
4. riavvia il kernel solo dopo aver conservato eventuali file che vuoi ispezionare;
5. non correggere un caso negativo: l'errore previsto è parte dell'esempio.